## Read from S3

Reads FRN 4.1 scores and Performance data from S3-mounted paths and prints samples.

In [ ]:
import pandas as pd
import os

#### Configuration

In [ ]:
# ── Configurable parameters ──────────────────────────────────────────
START_DATE  = "2016-06-27"   # Inclusive start of contract_date range
END_DATE    = "2021-06-27"   # Inclusive end of contract_date range
PERF_MONTHS = 48             # Performance window: 12, 18, 24, 36, or 48
EXCLUDE_CB  = True           # If True, exclude co-borrowers (loanpersonrole == 'CB')
MOB_POOL    = [ 18, 24, 36, 48]  # Months on book for multi-MOB default rate curve
SCORE_MIN   = 120             # Min model score for default-rate / marginal-impact charts
SCORE_MAX   = 160             # Max model score for default-rate / marginal-impact charts
# ─────────────────────────────────────────────────────────────────────

STATUS_COL = f"accountstatus_{PERF_MONTHS}"

# Seasoning check
end_dt = pd.to_datetime(END_DATE)
today  = pd.Timestamp.today()
months_seasoned = (today.year - end_dt.year) * 12 + (today.month - end_dt.month)

print(f"Configuration:")
print(f"  Contract date range : {START_DATE} to {END_DATE}")
print(f"  Performance window  : {PERF_MONTHS} months ({STATUS_COL})")
print(f"  Exclude CB          : {EXCLUDE_CB}")
print(f"  MOB pool            : {MOB_POOL}")
print(f"  Score range         : {SCORE_MIN} to {SCORE_MAX}")
print(f"  Months since END    : ~{months_seasoned}")
print()

if months_seasoned < PERF_MONTHS:
    shortfall = PERF_MONTHS - months_seasoned
    print(f"WARNING: Loans contracted up to {END_DATE} have at most ~{months_seasoned} months")
    print(f"  of seasoning as of today ({today.date()}), but {PERF_MONTHS}-month performance")
    print(f"  was requested. Results WILL include unseasoned loans (~{shortfall} months short).")
    print(f"  Consider setting END_DATE to {(today - pd.DateOffset(months=PERF_MONTHS)).strftime('%Y-%m-%d')} or earlier.")
else:
    print(f"Seasoning check PASSED: ~{months_seasoned} months available >= {PERF_MONTHS} months requested.")

#### Source Paths

In [ ]:
FRN_SCORES_PATH = r"S:\rds\rds_adhoc_rule_thumb_eval\data\archive_borrowers_prod.parquet"
MTN_SCORES_PATH = r"S:\rds\rds_adhoc_model_scores\MTN4.2\archive_data_2016_02_to_2025_06_scores.parquet"
PERFORMANCE_PATH = r"S:\rds\originations\2025\asa\base_file_2026_08.pipe.txt.gz"

for label, path in [("FRN 4.1 Scores", FRN_SCORES_PATH), ("MTN 4.2 Scores", MTN_SCORES_PATH), ("Performance", PERFORMANCE_PATH)]:
    exists = os.path.exists(path)
    print(f"{label}: {path}")
    print(f"  Exists: {exists}")
    if exists:
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  Size:   {size_mb:.2f} MB")
    print()

#### FRN 4.1 Scores (Parquet)

In [ ]:
df_frn = pd.read_parquet(FRN_SCORES_PATH)
print(f"Shape: {df_frn.shape[0]:,} rows x {df_frn.shape[1]} cols")
print(f"\nColumns: {list(df_frn.columns)}")
print(f"\nDtypes:\n{df_frn.dtypes}")
df_frn.head(10)

#### Performance (Pipe-delimited, gzipped)

In [ ]:
df_perf = pd.read_csv(PERFORMANCE_PATH, sep="|", compression="gzip", low_memory=False)
print(f"Shape: {df_perf.shape[0]:,} rows x {df_perf.shape[1]} cols")
print(f"\nColumns: {list(df_perf.columns)}")
print(f"\nDtypes:\n{df_perf.dtypes}")
df_perf.head(10)

#### MTN 4.2 Scores (Parquet)

In [ ]:
df_mtn = pd.read_parquet(MTN_SCORES_PATH)
print(f"Shape: {df_mtn.shape[0]:,} rows x {df_mtn.shape[1]} cols")
print(f"\nColumns: {list(df_mtn.columns)}")
print(f"\nDtypes:\n{df_mtn.dtypes}")
df_mtn.head(10)

#### Coverage Analysis: FRN 4.1 and MTN 4.2 vs Performance

In [ ]:
# ── Coverage: what % of Performance rows are in FRN / MTN Scores? ─
# FRN joins on dmsid; MTN joins on dms_account_num
frn_dmsids  = set(df_frn["dmsid"].dropna().unique())
perf_dmsids = set(df_perf["dmsid"].dropna().unique())

mtn_accts_set = set(df_mtn["dms_account_num"].dropna().unique())

perf_total_rows  = len(df_perf)

# Row-level (FRN) — join key is dmsid
perf_in_frn_mask = df_perf["dmsid"].isin(frn_dmsids)
frn_row_covered  = perf_in_frn_mask.sum()
frn_row_pct      = frn_row_covered / perf_total_rows * 100

# Row-level (MTN) — join key is dms_account_num
perf_in_mtn_mask = df_perf["dms_account_num"].isin(mtn_accts_set)
mtn_row_covered  = perf_in_mtn_mask.sum()
mtn_row_pct      = mtn_row_covered / perf_total_rows * 100

# Row-level (Combined)
combined_mask    = perf_in_frn_mask | perf_in_mtn_mask
comb_row_covered = combined_mask.sum()
comb_row_pct     = comb_row_covered / perf_total_rows * 100

# Unique-dmsid level (FRN only — MTN does not have dmsid)
unique_perf        = len(perf_dmsids)
frn_dmsid_overlap  = len(perf_dmsids & frn_dmsids)

# Account-level (both models have dms_account_num)
frn_accts  = set(df_frn["dms_account_num"].dropna().unique())
mtn_accts  = set(df_mtn["dms_account_num"].dropna().unique())
perf_accts = set(df_perf["dms_account_num"].dropna().unique())
frn_acct_overlap  = len(perf_accts & frn_accts)
mtn_acct_overlap  = len(perf_accts & mtn_accts)
comb_acct_overlap = len(perf_accts & (frn_accts | mtn_accts))

for model_label, row_covered, row_pct, acct_ovlp in [
    ("FRN 4.1",              frn_row_covered, frn_row_pct, frn_acct_overlap),
    ("MTN 4.2",              mtn_row_covered, mtn_row_pct, mtn_acct_overlap),
    ("COMBINED (FRN | MTN)", comb_row_covered, comb_row_pct, comb_acct_overlap),
]:
    print("=" * 60)
    print(f"COVERAGE: {model_label}  ->  Performance")
    print("=" * 60)
    print()
    print(f"  Perf rows    : {perf_total_rows:>12,} rows")
    print()
    print(f"  Row-level    : {row_covered:>10,} / {perf_total_rows:>10,}  ({row_pct:.2f}%)")
    acct_pct = acct_ovlp / len(perf_accts) * 100
    print(f"  Account-level: {acct_ovlp:>10,} / {len(perf_accts):>10,}  ({acct_pct:.2f}%)")
    print()

# Additional FRN-only dmsid-level coverage
dmsid_pct = frn_dmsid_overlap / unique_perf * 100
print(f"FRN 4.1 dmsid-level coverage: {frn_dmsid_overlap:,} / {unique_perf:,} ({dmsid_pct:.2f}%)")

#### Latest Contract Date

In [ ]:
frn_max = pd.to_datetime(df_frn["contract_date"]).max()
mtn_max = pd.to_datetime(df_mtn["con_date"]).max()
perf_max = pd.to_datetime(df_perf["contract_date"]).max()

print(f"FRN 4.1 Scores  - latest contract_date: {frn_max.date()}")
print(f"MTN 4.2 Scores  - latest con_date     : {mtn_max.date()}")
print(f"Performance     - latest contract_date: {perf_max.date()}")

#### Distinct Values — accountstatus_18

In [ ]:
print("Distinct accountstatus_18 values:")
print(df_perf["accountstatus_18"].unique())
print(f"\nCount: {df_perf['accountstatus_18'].nunique()} distinct values")

#### Default Rate by Model Score — FRN 4.1 vs MTN 4.2

In [ ]:
# Investigate snapshot_48mob to see how far the 48-month performance data extends
snapshot_col = "snapshot_48mob"

df_perf["_snap48"] = pd.to_datetime(df_perf[snapshot_col], errors="coerce")

print(f"=== {snapshot_col} investigation ===")
print(f"Non-null count : {df_perf['_snap48'].notna().sum():>10,} / {len(df_perf):>10,}")
print(f"Null/empty     : {df_perf['_snap48'].isna().sum():>10,}")
print(f"Min date       : {df_perf['_snap48'].min()}")
print(f"Max date       : {df_perf['_snap48'].max()}")
print()

# Latest contract_date for rows that actually HAVE a 48-month snapshot
df_perf["_cd"] = pd.to_datetime(df_perf["contract_date"], errors="coerce")
has_48 = df_perf["_snap48"].notna()

print(f"Rows WITH {snapshot_col} populated : {has_48.sum():>10,}")
print(f"  Latest contract_date among those  : {df_perf.loc[has_48, '_cd'].max()}")
print(f"  Earliest contract_date among those: {df_perf.loc[has_48, '_cd'].min()}")
print()

no_48 = ~has_48
print(f"Rows WITHOUT {snapshot_col}          : {no_48.sum():>10,}")
print(f"  Latest contract_date among those  : {df_perf.loc[no_48, '_cd'].max()}")
print(f"  Earliest contract_date among those: {df_perf.loc[no_48, '_cd'].min()}")

# Clean up temp columns
df_perf.drop(columns=["_snap48", "_cd"], inplace=True, errors="ignore")

In [ ]:
# Filter all datasets to the configured contract_date range
df_frn["contract_date_dt"]  = pd.to_datetime(df_frn["contract_date"])
df_mtn["contract_date_dt"]  = pd.to_datetime(df_mtn["con_date"])
df_perf["contract_date_dt"] = pd.to_datetime(df_perf["contract_date"])

mask_frn  = (df_frn["contract_date_dt"] >= START_DATE)  & (df_frn["contract_date_dt"] <= END_DATE)
mask_mtn  = (df_mtn["contract_date_dt"] >= START_DATE)  & (df_mtn["contract_date_dt"] <= END_DATE)
mask_perf = (df_perf["contract_date_dt"] >= START_DATE) & (df_perf["contract_date_dt"] <= END_DATE)

# Optionally exclude co-borrowers
if EXCLUDE_CB:
    perf_role_col = df_perf["loanpersonrole"].str.upper().str.strip()
    mask_perf = mask_perf & (perf_role_col != "CB")
    # MTN uses cb_flag (1 = coborrower)
    mask_mtn = mask_mtn & (df_mtn["cb_flag"] != 1)
    print(f"EXCLUDE_CB is ON: filtering performance data to loanpersonrole != 'CB'")
    print(f"EXCLUDE_CB is ON: filtering MTN data to cb_flag != 1")

df_frn_f  = df_frn[mask_frn].copy()
df_mtn_f  = df_mtn[mask_mtn].copy()
df_perf_f = df_perf[mask_perf].copy()

print(f"FRN 4.1 Scores : {len(df_frn):>10,} -> {len(df_frn_f):>10,} rows")
print(f"MTN 4.2 Scores : {len(df_mtn):>10,} -> {len(df_mtn_f):>10,} rows")
print(f"Performance    : {len(df_perf):>10,} -> {len(df_perf_f):>10,} rows")
print(f"Date range     : {df_perf_f['contract_date_dt'].min().date()} to {df_perf_f['contract_date_dt'].max().date()}")

In [ ]:
# Normalize the selected status column to lowercase
print(f"Using column: {STATUS_COL}")
df_perf_f["status_lower"] = df_perf_f[STATUS_COL].str.lower().str.strip()

# Categorize: default, non-default, or exclude
def categorize(status):
    if pd.isna(status):
        return "exclude"        # nan -> exclude
    if status in ("chargeoff", "settled"):
        return "default"
    if status == "buyback":
        return "exclude"        # buyback -> exclude
    return "non-default"        # paid, active -> non-default

df_perf_f["default_flag"] = df_perf_f["status_lower"].apply(categorize)

print("\nCategory counts:")
print(df_perf_f["default_flag"].value_counts())
print(f"\nExcluding {(df_perf_f['default_flag'] == 'exclude').sum():,} rows (nan + buyback)")

In [ ]:
# Filter to default / non-default only (exclude nan and buyback)
df_perf_valid = df_perf_f[df_perf_f["default_flag"] != "exclude"].copy()
df_perf_valid["is_default"] = (df_perf_valid["default_flag"] == "default").astype(int)

print(f"Rows after exclusions: {len(df_perf_valid):,}")
print(f"Defaults:     {df_perf_valid['is_default'].sum():,}")
print(f"Non-defaults: {(df_perf_valid['is_default'] == 0).sum():,}")
print(f"Overall {PERF_MONTHS}-month default rate: {df_perf_valid['is_default'].mean():.4%}")

In [ ]:
# Join performance (with default flag) to FRN 4.1 scores on dmsid
df_merged = df_perf_valid[["dmsid", "is_default"]].merge(
    df_frn_f[["dmsid", "model_score"]],
    on="dmsid",
    how="inner"
)

print(f"FRN 4.1 Merged rows: {len(df_merged):,}")
print(f"FRN 4.1 Unique dmsid: {df_merged['dmsid'].nunique():,}")
df_merged.head()

In [ ]:
# Round model_score to nearest integer and group
df_merged["score_bucket"] = df_merged["model_score"].round(0).astype(int)

default_by_score = (
    df_merged
    .groupby("score_bucket")
    .agg(count=("is_default", "size"), defaults=("is_default", "sum"))
    .reset_index()
)
default_by_score["default_rate"] = default_by_score["defaults"] / default_by_score["count"]
default_by_score = default_by_score.sort_values("score_bucket")

print(f"Distinct score buckets: {len(default_by_score)}")
default_by_score

In [ ]:
# Join performance (with default flag) to MTN 4.2 scores on dms_account_num
df_merged_mtn = df_perf_valid[["dms_account_num", "is_default"]].merge(
    df_mtn_f[["dms_account_num", "mtn4_2_score"]],
    on="dms_account_num",
    how="inner"
)

print(f"MTN 4.2 Merged rows: {len(df_merged_mtn):,}")
print(f"MTN 4.2 Unique accounts: {df_merged_mtn['dms_account_num'].nunique():,}")
df_merged_mtn.head()

In [ ]:
# Round mtn4_2_score to nearest integer and group (MTN 4.2)
df_merged_mtn["score_bucket"] = df_merged_mtn["mtn4_2_score"].round(0).astype(int)

default_by_score_mtn = (
    df_merged_mtn
    .groupby("score_bucket")
    .agg(count=("is_default", "size"), defaults=("is_default", "sum"))
    .reset_index()
)
default_by_score_mtn["default_rate"] = default_by_score_mtn["defaults"] / default_by_score_mtn["count"]
default_by_score_mtn = default_by_score_mtn.sort_values("score_bucket")

print(f"MTN 4.2 Distinct score buckets: {len(default_by_score_mtn)}")
default_by_score_mtn

#### Score Distribution (Loan Count by Score Bucket, 100-160) — FRN 4.1 vs MTN 4.2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Filter to 100-160 range for both models
mask_dist_frn = (default_by_score["score_bucket"] >= 100) & (default_by_score["score_bucket"] <= 160)
df_dist_frn = default_by_score[mask_dist_frn].copy().sort_values("score_bucket")

mask_dist_mtn = (default_by_score_mtn["score_bucket"] >= 100) & (default_by_score_mtn["score_bucket"] <= 160)
df_dist_mtn = default_by_score_mtn[mask_dist_mtn].copy().sort_values("score_bucket")

fig, (ax_frn, ax_mtn) = plt.subplots(1, 2, figsize=(22, 5), sharey=True)

# FRN 4.1
ax_frn.bar(df_dist_frn["score_bucket"], df_dist_frn["count"], color="steelblue", alpha=0.8, width=0.9)
ax_frn.set_xlabel("Model Score (rounded to integer)")
ax_frn.set_ylabel("Loan Count")
ax_frn.set_title(f"FRN 4.1 — Score Distribution ({PERF_MONTHS}-Month, {START_DATE} to {END_DATE})\nScore Range 100-160")
ax_frn.set_xlim(98, 162)
ax_frn.grid(True, alpha=0.3, axis="y")
total_frn = df_dist_frn["count"].sum()
ax_frn.text(0.98, 0.95, f"Total loans (100-160): {total_frn:,}",
        transform=ax_frn.transAxes, ha="right", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.7))

# MTN 4.2
ax_mtn.bar(df_dist_mtn["score_bucket"], df_dist_mtn["count"], color="darkorange", alpha=0.8, width=0.9)
ax_mtn.set_xlabel("Model Score (rounded to integer)")
ax_mtn.set_title(f"MTN 4.2 — Score Distribution ({PERF_MONTHS}-Month, {START_DATE} to {END_DATE})\nScore Range 100-160")
ax_mtn.set_xlim(98, 162)
ax_mtn.grid(True, alpha=0.3, axis="y")
total_mtn = df_dist_mtn["count"].sum()
ax_mtn.text(0.98, 0.95, f"Total loans (100-160): {total_mtn:,}",
        transform=ax_mtn.transAxes, ha="right", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.7))

plt.tight_layout()
plt.show()

print(f"\nFRN 4.1 — Loans in 100-160 range: {total_frn:,} / {df_merged.shape[0]:,} total ({total_frn / df_merged.shape[0]:.1%})")
print(f"MTN 4.2 — Loans in 100-160 range: {total_mtn:,} / {df_merged_mtn.shape[0]:,} total ({total_mtn / df_merged_mtn.shape[0]:.1%})")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def _fit_and_plot(ax, default_by_score_df, label, colour_scatter, colour_line, score_min, score_max, perf_months, start_date, end_date):
    mask_range = (default_by_score_df["score_bucket"] >= score_min) & (default_by_score_df["score_bucket"] <= score_max)
    df_plot = default_by_score_df[mask_range].copy().sort_values("score_bucket")
    x = df_plot["score_bucket"].values.astype(float)
    y = df_plot["default_rate"].values
    best_degree, best_sse = 2, np.inf
    for deg in range(2, 9):
        coeffs = np.polyfit(x, y, deg)
        fitted = np.polyval(coeffs, x)
        sse = np.sum((y - fitted) ** 2)
        n = len(x)
        bic = n * np.log(sse / n + 1e-15) + deg * np.log(n)
        if bic < best_sse:
            best_sse = bic
            best_degree = deg
    coeffs = np.polyfit(x, y, best_degree)
    poly = np.poly1d(coeffs)
    x_smooth = np.linspace(score_min, score_max, 300)
    y_smooth = poly(x_smooth)
    ax.scatter(x, y, color=colour_scatter, s=40, zorder=3, label="Observed default rate")
    ax.plot(x_smooth, y_smooth, color=colour_line, linewidth=2, label=f"Best fit (degree {best_degree})")
    ax.set_xlabel("Model Score (rounded to integer)")
    ax.set_ylabel("Default Rate")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax.set_title(f"{label} — Default Rate by Score ({perf_months}-Month, {start_date} to {end_date})\nScore Range {score_min}-{score_max}")
    ax.set_xlim(score_min - 2, score_max + 2)
    ax.legend()
    ax.grid(True, alpha=0.3)
    return poly

fig, (ax_frn, ax_mtn) = plt.subplots(1, 2, figsize=(24, 5), sharey=True)

poly = _fit_and_plot(ax_frn, default_by_score, "FRN 4.1", "crimson", "steelblue", SCORE_MIN, SCORE_MAX, PERF_MONTHS, START_DATE, END_DATE)
poly_mtn = _fit_and_plot(ax_mtn, default_by_score_mtn, "MTN 4.2", "crimson", "darkorange", SCORE_MIN, SCORE_MAX, PERF_MONTHS, START_DATE, END_DATE)

poly_deriv     = poly.deriv()
poly_deriv_mtn = poly_mtn.deriv()

plt.tight_layout()
plt.show()

#### Default Rate by Model Score - Multi-MOB Comparison — FRN 4.1 vs MTN 4.2

Overlays the default rate by model score curve for each month-on-book (MOB) horizon in `MOB_POOL`. Uses the same `START_DATE` / `END_DATE` window and score range `SCORE_MIN`-`SCORE_MAX`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── colour palette for MOB curves ──────────────────────────────────────
mob_colours = {
    12: "#1f77b4",
    18: "#ff7f0e",
    24: "#2ca02c",
    36: "#d62728",
    48: "#9467bd",
}

def _plot_multi_mob_default_rate(ax, score_df, model_label, perf_df, score_min, score_max, start_date, end_date, join_key="dmsid", score_col="model_score"):
    for mob in sorted(MOB_POOL):
        status_col_mob = f"accountstatus_{mob}"
        if status_col_mob not in perf_df.columns:
            print(f"WARNING: column {status_col_mob} not found in performance data -- skipping MOB {mob}")
            continue
        status_lower_mob = perf_df[status_col_mob].str.lower().str.strip()
        def _categorise(s):
            if pd.isna(s):
                return "exclude"
            if s in ("chargeoff", "settled"):
                return "default"
            if s == "buyback":
                return "exclude"
            return "non-default"
        flag_mob = status_lower_mob.apply(_categorise)
        df_valid_mob = perf_df[flag_mob != "exclude"].copy()
        df_valid_mob["is_default"] = (flag_mob[flag_mob != "exclude"] == "default").astype(int)
        df_merged_mob = df_valid_mob[[join_key, "is_default"]].merge(
            score_df[[join_key, score_col]],
            on=join_key,
            how="inner",
        )
        if df_merged_mob.empty:
            print(f"WARNING: no merged data for MOB {mob} -- skipping")
            continue
        df_merged_mob["score_bucket"] = df_merged_mob[score_col].round(0).astype(int)
        agg_mob = (
            df_merged_mob
            .groupby("score_bucket")
            .agg(count=("is_default", "size"), defaults=("is_default", "sum"))
            .reset_index()
        )
        agg_mob["default_rate"] = agg_mob["defaults"] / agg_mob["count"]
        mask_r = (agg_mob["score_bucket"] >= score_min) & (agg_mob["score_bucket"] <= score_max)
        df_p = agg_mob[mask_r].copy().sort_values("score_bucket")
        if df_p.empty:
            continue
        xv = df_p["score_bucket"].values.astype(float)
        yv = df_p["default_rate"].values
        best_deg_mob, best_bic_mob = 2, np.inf
        for d in range(2, 9):
            c = np.polyfit(xv, yv, d)
            fitted = np.polyval(c, xv)
            sse = np.sum((yv - fitted) ** 2)
            n = len(xv)
            bic = n * np.log(sse / n + 1e-15) + d * np.log(n)
            if bic < best_bic_mob:
                best_bic_mob = bic
                best_deg_mob = d
        coeffs_mob = np.polyfit(xv, yv, best_deg_mob)
        poly_mob   = np.poly1d(coeffs_mob)
        x_smooth   = np.linspace(score_min, score_max, 300)
        y_smooth   = poly_mob(x_smooth)
        colour = mob_colours.get(mob, None)
        overall_dr = df_valid_mob["is_default"].mean()
        lbl = f"MOB {mob}  (overall: {overall_dr:.2%})"
        ax.scatter(xv, yv, color=colour, s=30, alpha=0.5, zorder=3)
        ax.plot(x_smooth, y_smooth, color=colour, linewidth=2, label=lbl, zorder=2)
    ax.set_xlabel("Model Score (rounded to integer)")
    ax.set_ylabel("Default Rate")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax.set_title(
        f"{model_label} — Default Rate Multi-MOB Comparison\n"
        f"({start_date} to {end_date}, Score Range {score_min}-{score_max})"
    )
    ax.set_xlim(score_min - 2, score_max + 2)
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)

fig, (ax_frn, ax_mtn) = plt.subplots(1, 2, figsize=(28, 6), sharey=True)

_plot_multi_mob_default_rate(ax_frn, df_frn_f, "FRN 4.1", df_perf_f, SCORE_MIN, SCORE_MAX, START_DATE, END_DATE)
_plot_multi_mob_default_rate(ax_mtn, df_mtn_f, "MTN 4.2", df_perf_f, SCORE_MIN, SCORE_MAX, START_DATE, END_DATE, join_key="dms_account_num", score_col="mtn4_2_score")

plt.tight_layout()
plt.show()

#### FRN 4.1 vs MTN 4.2 — Default Rate Comparison at PERF_MONTHS

Single overlay of both model curves at the configured `PERF_MONTHS` horizon for direct comparison.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(14, 6))

# ── FRN 4.1 ──────────────────────────────────────────────────────────
mask_frn = (
    (default_by_score["score_bucket"] >= SCORE_MIN)
    & (default_by_score["score_bucket"] <= SCORE_MAX)
)
df_frn_plot = default_by_score[mask_frn].copy().sort_values("score_bucket")
x_frn = df_frn_plot["score_bucket"].values.astype(float)
y_frn = df_frn_plot["default_rate"].values

best_deg_frn, best_bic_frn = 2, np.inf
for d in range(2, 9):
    c = np.polyfit(x_frn, y_frn, d)
    fitted = np.polyval(c, x_frn)
    sse = np.sum((y_frn - fitted) ** 2)
    n = len(x_frn)
    bic = n * np.log(sse / n + 1e-15) + d * np.log(n)
    if bic < best_bic_frn:
        best_bic_frn = bic
        best_deg_frn = d

coeffs_frn = np.polyfit(x_frn, y_frn, best_deg_frn)
poly_frn   = np.poly1d(coeffs_frn)
x_smooth   = np.linspace(SCORE_MIN, SCORE_MAX, 300)

overall_dr_frn = df_perf_valid.merge(
    df_frn_f[["dmsid"]], on="dmsid", how="inner"
)["is_default"].mean()

ax.scatter(x_frn, y_frn, color="steelblue", s=30, alpha=0.4, zorder=3)
ax.plot(
    x_smooth, poly_frn(x_smooth),
    color="steelblue", linewidth=2.5,
    label=f"FRN 4.1 ",
    zorder=4,
)

# ── MTN 4.2 ──────────────────────────────────────────────────────────
mask_mtn = (
    (default_by_score_mtn["score_bucket"] >= SCORE_MIN)
    & (default_by_score_mtn["score_bucket"] <= SCORE_MAX)
)
df_mtn_plot = default_by_score_mtn[mask_mtn].copy().sort_values("score_bucket")
x_mtn = df_mtn_plot["score_bucket"].values.astype(float)
y_mtn = df_mtn_plot["default_rate"].values

best_deg_mtn, best_bic_mtn = 2, np.inf
for d in range(2, 9):
    c = np.polyfit(x_mtn, y_mtn, d)
    fitted = np.polyval(c, x_mtn)
    sse = np.sum((y_mtn - fitted) ** 2)
    n = len(x_mtn)
    bic = n * np.log(sse / n + 1e-15) + d * np.log(n)
    if bic < best_bic_mtn:
        best_bic_mtn = bic
        best_deg_mtn = d

coeffs_mtn = np.polyfit(x_mtn, y_mtn, best_deg_mtn)
poly_mtn_cmp = np.poly1d(coeffs_mtn)

overall_dr_mtn = df_perf_valid.merge(
    df_mtn_f[["dms_account_num"]], on="dms_account_num", how="inner"
)["is_default"].mean()

ax.scatter(x_mtn, y_mtn, color="darkorange", s=30, alpha=0.4, zorder=3)
ax.plot(
    x_smooth, poly_mtn_cmp(x_smooth),
    color="darkorange", linewidth=2.5,
    label=f"MTN 4.2 ",
    zorder=4,
)

# ── formatting ───────────────────────────────────────────────────────
ax.set_xlabel("Model Score")
ax.set_ylabel("Unit Loss")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.set_xlim(SCORE_MIN - 2, SCORE_MAX + 2)
ax.set_title(
    f"FRN 4.1 vs MTN 4.2 tc Unit Loss at {PERF_MONTHS} MOB\n\n"
    f"On repriced {2016} to {2021} loans"
)
ax.legend(loc="upper right", fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Third-degree polynomial best fit ─────────────────────────────────
coeffs_frn_3 = np.polyfit(x_frn, y_frn, 3)
coeffs_mtn_3 = np.polyfit(x_mtn, y_mtn, 3)

def _fmt_poly3(coeffs, label):
    a, b, c, d = coeffs
    terms = []
    terms.append(f"{a:+.6e} x^3")
    terms.append(f"{b:+.6e} x^2")
    terms.append(f"{c:+.6e} x")
    terms.append(f"{d:+.6e}")
    equation = "  ".join(terms)
    return f"{label}:  y = {equation}"

print("Third-Degree Polynomial Best Fit (Default Rate as a function of Model Score):")
print()
print(_fmt_poly3(coeffs_frn_3, "FRN 4.1"))
print()
print(_fmt_poly3(coeffs_mtn_3, "MTN 4.2"))


#### FRN 4.1 — Default Rate and First Derivative by Model Score

Overlays the fitted default rate curve (steelblue, left axis) and its first derivative (purple, right axis) on a single plot. Data labels are shown at model scores 120, 140, and 150.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math

# ── Fit polynomial to FRN 4.1 default rate data ───────────────────────
mask_frn_dr = (
    (default_by_score["score_bucket"] >= SCORE_MIN)
    & (default_by_score["score_bucket"] <= SCORE_MAX)
)
df_frn_dr = default_by_score[mask_frn_dr].copy().sort_values("score_bucket")
x_frn_dr  = df_frn_dr["score_bucket"].values.astype(float)
y_frn_dr  = df_frn_dr["default_rate"].values

best_deg_frn_dr, best_bic_frn_dr = 2, np.inf
for _d in range(2, 9):
    _c = np.polyfit(x_frn_dr, y_frn_dr, _d)
    _fitted = np.polyval(_c, x_frn_dr)
    _sse = np.sum((y_frn_dr - _fitted) ** 2)
    _n   = len(x_frn_dr)
    _bic = _n * np.log(_sse / _n + 1e-15) + _d * np.log(_n)
    if _bic < best_bic_frn_dr:
        best_bic_frn_dr = _bic
        best_deg_frn_dr = _d

poly_frn_dr    = np.poly1d(np.polyfit(x_frn_dr, y_frn_dr, best_deg_frn_dr))
poly_frn_dr_d1 = poly_frn_dr.deriv()

x_smooth_dr = np.linspace(SCORE_MIN, SCORE_MAX, 500)
y_curve     = poly_frn_dr(x_smooth_dr)
y_deriv_dr  = poly_frn_dr_d1(x_smooth_dr)

# ── Aligned gridline axis ranges ──────────────────────────────────────
# -0.5 pct-pt derivative  <->  20 pct-pt default rate  (ratio = 40)
# Changing DR_STEP to 0.20 halves the derivative axis relative to DR.
DR_STEP = 0.20    # 20 % per gridline on the default-rate axis
D1_STEP = 0.005   # 0.5 pct-pt per gridline on the derivative axis (raw units)

# Cosmetic bottom padding — extends ylim downward so curves never touch the frame.
# Ticks are NOT added in this region, so labels remain clean.
DR_BOTTOM_PAD = DR_STEP * 0.5   # half a DR gridline step of extra whitespace
D1_BOTTOM_PAD = D1_STEP * 0.5   # matching padding on the derivative axis

# Steps needed to cover each dataset — no extra padding so axes don't
# extend below the actual data (math.ceil alone is sufficient).
_d1_raw_min  = float(np.min(y_deriv_dr))
_n_d1        = math.ceil(abs(_d1_raw_min) / D1_STEP)   # steps to cover derivative data

_dr_raw_max  = float(np.max(y_frn_dr))
_dr_raw_min  = float(np.min(y_frn_dr))
_dr_ylim_max = math.ceil(_dr_raw_max / DR_STEP) * DR_STEP
_n_dr        = math.ceil((_dr_ylim_max - _dr_raw_min) / DR_STEP)

# DR axis drives the step count: this pins the derivative axis to half its previous range
# (-0.5 % aligns with 60 %, -1.0 % with 40 %, etc.).
# Derivative data beyond the bottom gridline is plotted but outside the visible area.
_n_steps     = _n_dr

_d1_ylim_min = -_n_steps * D1_STEP          # stops here, not below
_dr_ylim_min =  _dr_ylim_max - _n_steps * DR_STEP  # stops at 0 %, not below

# Explicit tick arrays — same count guarantees gridlines lock together
_dr_ticks = np.arange(_dr_ylim_min, _dr_ylim_max + DR_STEP / 10, DR_STEP)
_d1_ticks = np.arange(_d1_ylim_min, D1_STEP / 10,               D1_STEP)

# ── Figure with twin axes ──────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(14, 6))
ax2 = ax1.twinx()

# Default rate — steelblue (left axis)
ax1.scatter(x_frn_dr, y_frn_dr, color="steelblue", s=30, alpha=0.4, zorder=3)
line_dr, = ax1.plot(
    x_smooth_dr, y_curve,
    color="steelblue", linewidth=2.5,
    label=f"FRN 4.1 Default Rate",
    zorder=4,
)

# First derivative — darkgreen (right axis)
line_d1, = ax2.plot(
    x_smooth_dr, y_deriv_dr,
    color="darkgreen", linewidth=2, linestyle="--",
    label="UL to MS Sloping (First Derivative)",
    zorder=4,
)

# ── Apply matched axis limits and ticks ───────────────────────────────
ax1.set_ylim(_dr_ylim_min - DR_BOTTOM_PAD, _dr_ylim_max)
ax2.set_ylim(_d1_ylim_min - D1_BOTTOM_PAD, 0)  # 0 at the top; no positive region
ax1.set_yticks(_dr_ticks)
ax2.set_yticks(_d1_ticks)
ax2.grid(False)                           # gridlines from ax1 only
ax2.axhline(0, color="darkgreen", linewidth=0.8, linestyle=":", alpha=0.5)

# ── Axis labels and formatters ─────────────────────────────────────────
ax1.set_xlabel("Model Score")
ax1.set_ylabel("Unit Loss", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
ax1.set_xlim(SCORE_MIN - 2, SCORE_MAX + 2)

ax2.set_ylabel("UL % sloping per additional MS", color="darkgreen")
ax2.tick_params(axis="y", labelcolor="darkgreen")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v * 100:.1f}%"))

# ── Data label offsets — edit these to reposition labels ──────────────
# All values are in POINTS (1 pt ≈ 1 px at 72 dpi).
# +x = right of marker   -x = left    +y = above marker   -y = below
#
#           score : (dr_dx, dr_dy,  d1_dx, d1_dy)
_lbl_offsets = {
    120:          (   12,   -80,      12,    30  ),  # DR below (near top); deriv above
    140:          (   12,    -50,      12,    50  ),  # both above-right
    150:          (   12,    50,      12,    -50  ),  # both above-right
}

for _s in [120, 140, 150]:
    _dr_val  = float(poly_frn_dr(_s))
    _d1_val  = float(poly_frn_dr_d1(_s))
    _dr_dx, _dr_dy, _d1_dx, _d1_dy = _lbl_offsets[_s]

    # Default rate marker + label
    ax1.plot(_s, _dr_val, "o", color="steelblue", markersize=8, zorder=5)
    ax1.annotate(
        f" Score {_s}\n {_dr_val:.1%} UL  ",
        xy=(_s, _dr_val), xycoords="data",
        xytext=(_dr_dx, _dr_dy), textcoords="offset points",
        fontsize=8.5, color="steelblue", fontweight="bold",
        ha="left" if _dr_dx >= 0 else "right",
        arrowprops=dict(arrowstyle="->", color="steelblue", lw=1.2),
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="steelblue", alpha=0.8),
    )

    # Derivative marker + label
    ax2.plot(_s, _d1_val, "o", color="darkgreen", markersize=8, zorder=5)
    ax2.annotate(
        f" Score {_s}\n {_d1_val * 100:.1f}% UL/MS ",
        xy=(_s, _d1_val), xycoords="data",
        xytext=(_d1_dx, _d1_dy), textcoords="offset points",
        fontsize=8.5, color="darkgreen", fontweight="bold",
        ha="left" if _d1_dx >= 0 else "right",
        arrowprops=dict(arrowstyle="->", color="darkgreen", lw=1.2),
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="darkgreen", alpha=0.8),
    )

# ── Title and legend ───────────────────────────────────────────────────
ax1.set_title(
    f"tc Unit Loss by Model Score & Unit Loss sloping \n\n"
    f"On repriced FRN 4.1, {2016} to {2021}, {PERF_MONTHS}-Month Performance "
    # f"Score Range {SCORE_MIN}-{SCORE_MAX})"
)
lines_combined  = [line_dr, line_d1]
labels_combined = [l.get_label() for l in lines_combined]
ax1.legend(lines_combined, labels_combined, loc="upper right", fontsize=10)
ax1.grid(True, alpha=0.3)

# ── Horizontal reference lines at 10%, 30%, 50%, 70% default rate ───────
# Style matches the background grid (same colour, weight and alpha).
_grid_props = dict(
    color='#b0b0b0', linewidth=0.8, linestyle='-', alpha=0.3, zorder=1
)
for _ref_lvl in [0.10, 0.30, 0.50, 0.70]:
    ax1.axhline(_ref_lvl, **_grid_props)

plt.tight_layout()
plt.show()


#### Marginal Impact: Change in Default Rate per +1 Score Point — FRN 4.1 vs MTN 4.2

At each 10-score band, the derivative of the fitted curve tells us how much the default rate changes for every additional model score point.

In [ ]:
# Derivative (slope) at every 5-score band within configured range
score_bands = np.arange(SCORE_MIN, SCORE_MAX + 1, 5)

rows_frn = []
for s in score_bands:
    fitted_rate = poly(s)
    slope = poly_deriv(s)
    slope_pct = slope * 100
    rows_frn.append({
        "Model Score": int(s),
        "Fitted Default Rate": f"{fitted_rate:.2%}",
        "dRate/dScore (% per +1 pt)": f"{slope_pct:+.2f}%",
    })

df_deriv_frn = pd.DataFrame(rows_frn)
print("FRN 4.1 — Marginal impact of +1 model score point on default rate (from fitted curve):")
print()
display(df_deriv_frn)

rows_mtn = []
for s in score_bands:
    fitted_rate = poly_mtn(s)
    slope = poly_deriv_mtn(s)
    slope_pct = slope * 100
    rows_mtn.append({
        "Model Score": int(s),
        "Fitted Default Rate": f"{fitted_rate:.2%}",
        "dRate/dScore (% per +1 pt)": f"{slope_pct:+.2f}%",
    })

df_deriv_mtn = pd.DataFrame(rows_mtn)
print("\nMTN 4.2 — Marginal impact of +1 model score point on default rate (from fitted curve):")
print()
display(df_deriv_mtn)

In [ ]:
# Plot: Model Score vs Derivative (marginal change in default rate per +1 score point)

fig, (ax_frn, ax_mtn) = plt.subplots(1, 2, figsize=(24, 5), sharey=True)

for ax, poly_d, label, line_colour in [(ax_frn, poly_deriv, "FRN 4.1", "steelblue"), (ax_mtn, poly_deriv_mtn, "MTN 4.2", "darkorange")]:
    x_deriv_smooth = np.linspace(SCORE_MIN, SCORE_MAX, 300)
    y_deriv_smooth = poly_d(x_deriv_smooth) * 100
    ax.plot(x_deriv_smooth, y_deriv_smooth, color=line_colour, linewidth=2, label="Derivative (fitted curve)")
    band_scores = np.arange(SCORE_MIN, SCORE_MAX + 1, 5)
    band_derivs = poly_d(band_scores) * 100
    ax.scatter(band_scores, band_derivs, color="crimson", s=60, zorder=3, label="5-score bands")
    for s, d in zip(band_scores, band_derivs):
        ax.annotate(f"{d:+.2f}%", (s, d), textcoords="offset points",
                    xytext=(0, 12), ha="center", fontsize=9, fontweight="bold")
    ax.axhline(y=0, color="gray", linestyle="--", linewidth=0.8)
    ax.set_xlabel("Model Score")
    ax.set_ylabel("Change in Default Rate per +1 Score Point (%)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:+.1f}%"))
    ax.set_title(f"{label} — Marginal Impact ({PERF_MONTHS}-Month, {START_DATE} to {END_DATE})\nScore Range {SCORE_MIN}-{SCORE_MAX}")
    ax.set_xlim(SCORE_MIN - 2, SCORE_MAX + 2)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print the degree-2 line of best fit for each model's derivative curve
for label, poly_d in [("FRN 4.1", poly_deriv), ("MTN 4.2", poly_deriv_mtn)]:
    # Sample the derivative at every integer score in the range
    x_pts = np.arange(SCORE_MIN, SCORE_MAX + 1).astype(float)
    y_pts = poly_d(x_pts) * 100  # in pct-pt units
    # Fit a degree-2 polynomial to the derivative values
    c2 = np.polyfit(x_pts, y_pts, 2)
    print(f"\n{label} — Degree-2 best fit of derivative (dRate/dScore in %):")
    print(f"  f(x) = {c2[0]:+.6f} x^2  {c2[1]:+.6f} x  {c2[2]:+.4f}")
    print(f"  Coefficients: a={c2[0]:.6f}, b={c2[1]:.6f}, c={c2[2]:.4f}")

#### Marginal Impact of Model Score - Multi-MOB Comparison — FRN 4.1 vs MTN 4.2

Overlays the derivative (change in default rate per +1 model score point) for each MOB horizon in `MOB_POOL`, using the same `START_DATE` / `END_DATE` window and score range `SCORE_MIN`-`SCORE_MAX`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

mob_colours = {
    12: "#1f77b4",
    18: "#ff7f0e",
    24: "#2ca02c",
    36: "#d62728",
    48: "#9467bd",
}

def _plot_multi_mob_marginal(ax, score_df, model_label, perf_df, score_min, score_max, start_date, end_date, join_key="dmsid", score_col="model_score"):
    for mob in sorted(MOB_POOL):
        status_col_mob = f"accountstatus_{mob}"
        if status_col_mob not in perf_df.columns:
            print(f"WARNING: column {status_col_mob} not found -- skipping MOB {mob}")
            continue
        status_lower_mob = perf_df[status_col_mob].str.lower().str.strip()
        def _categorise(s):
            if pd.isna(s):
                return "exclude"
            if s in ("chargeoff", "settled"):
                return "default"
            if s == "buyback":
                return "exclude"
            return "non-default"
        flag_mob = status_lower_mob.apply(_categorise)
        df_valid_mob = perf_df[flag_mob != "exclude"].copy()
        df_valid_mob["is_default"] = (flag_mob[flag_mob != "exclude"] == "default").astype(int)
        df_merged_mob = df_valid_mob[[join_key, "is_default"]].merge(
            score_df[[join_key, score_col]],
            on=join_key,
            how="inner",
        )
        if df_merged_mob.empty:
            continue
        df_merged_mob["score_bucket"] = df_merged_mob[score_col].round(0).astype(int)
        agg_mob = (
            df_merged_mob
            .groupby("score_bucket")
            .agg(count=("is_default", "size"), defaults=("is_default", "sum"))
            .reset_index()
        )
        agg_mob["default_rate"] = agg_mob["defaults"] / agg_mob["count"]
        mask_r = (agg_mob["score_bucket"] >= score_min) & (agg_mob["score_bucket"] <= score_max)
        df_p = agg_mob[mask_r].copy().sort_values("score_bucket")
        if df_p.empty:
            continue
        xv = df_p["score_bucket"].values.astype(float)
        yv = df_p["default_rate"].values
        best_deg_mob, best_bic_mob = 2, np.inf
        for d in range(2, 9):
            c = np.polyfit(xv, yv, d)
            fitted = np.polyval(c, xv)
            sse = np.sum((yv - fitted) ** 2)
            n = len(xv)
            bic = n * np.log(sse / n + 1e-15) + d * np.log(n)
            if bic < best_bic_mob:
                best_bic_mob = bic
                best_deg_mob = d
        poly_mob       = np.poly1d(np.polyfit(xv, yv, best_deg_mob))
        poly_deriv_mob = poly_mob.deriv()
        x_smooth     = np.linspace(score_min, score_max, 300)
        y_deriv_mob  = poly_deriv_mob(x_smooth) * 100
        colour = mob_colours.get(mob, None)
        ax.plot(x_smooth, y_deriv_mob, color=colour, linewidth=2, label=f"MOB {mob}", zorder=2)
        band_scores = np.arange(score_min, score_max + 1, 5)
        band_derivs = poly_deriv_mob(band_scores) * 100
        ax.scatter(band_scores, band_derivs, color=colour, s=40, alpha=0.7, zorder=3)
    ax.axhline(y=0, color="gray", linestyle="--", linewidth=0.8)
    ax.set_xlabel("Model Score")
    ax.set_ylabel("Change in Default Rate per +1 Score Point (%)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:+.1f}%"))
    ax.set_title(
        f"{model_label} — Marginal Impact Multi-MOB Comparison\n"
        f"({start_date} to {end_date}, Score Range {score_min}-{score_max})"
    )
    ax.set_xlim(score_min - 2, score_max + 2)
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)

fig, (ax_frn, ax_mtn) = plt.subplots(1, 2, figsize=(28, 6), sharey=True)

_plot_multi_mob_marginal(ax_frn, df_frn_f, "FRN 4.1", df_perf_f, SCORE_MIN, SCORE_MAX, START_DATE, END_DATE)
_plot_multi_mob_marginal(ax_mtn, df_mtn_f, "MTN 4.2", df_perf_f, SCORE_MIN, SCORE_MAX, START_DATE, END_DATE, join_key="dms_account_num", score_col="mtn4_2_score")

plt.tight_layout()
plt.show()